# Pretrained Fusion: Late Logits

Self-contained notebook for the late-fusion pretrained variant.

In [1]:
# Dataset root - edit this first if your Kaggle input path changes.
DATASET_ROOT = "/kaggle/input/datasets/anhduy54/visual-audio/raw_dataset"
OUTPUT_DIR = "/kaggle/working/outputs/variants"
FUSION_KIND = "late"
SEED = 42
EPOCHS = 20
BATCH_SIZE = 4
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
EARLY_STOPPING_PATIENCE = 10
MIN_DELTA = 1e-4

import json
import math
import random
import wave
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

LABELS = ("ambient", "leaf", "trunk", "twig")
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
PAPER_AVERAGE = "weighted"

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

@dataclass
class DataConfig:
    data_root: Path
    target_sample_rate: int = 16000
    audio_window_sec: float = 0.8
    n_mels: int = 128
    n_fft: int = 1024
    hop_length: int = 256
    image_size: int = 224
    skip_missing_files: bool = True
    train_crop: str = "random"
    eval_crop: str = "energy"
    normalize_audio_db: bool = True
    spectral_gate: bool = True
    spectral_gate_noise_percentile: float = 20.0
    spectral_gate_strength: float = 1.0

@dataclass
class ModelConfig:
    ast_model_name: str = "MIT/ast-finetuned-audioset-10-10-0.4593"
    clap_model_name: str = "laion/clap-htsat-unfused"
    fusion_dim: int = 256
    fusion_heads: int = 4
    fusion_layers: int = 2
    fusion_dropout: float = 0.1
    freeze_pretrained: bool = True
    ast_input_source: str = "mel"

def find_data_root(data_root):
    data_root = Path(data_root)
    for candidate in [data_root, data_root / "raw_dataset", data_root / "prepared_data", data_root / "dataset", Path("/kaggle/input/datasets/anhduy54/visual-audio/raw_dataset"), Path("/kaggle/input/visual-audio/raw_dataset")]:
        if (candidate / "audio_visual_dataset_default" / "dataset.csv").exists():
            return candidate
    return data_root

def build_index(data_root, skip_missing_files=True):
    rows = []
    for split_name in ("audio_visual_dataset_default", "audio_visual_dataset_robo_default"):
        split_dir = Path(data_root) / split_name
        csv_path = split_dir / "dataset.csv"
        if not csv_path.exists():
            continue
        for item in pd.read_csv(csv_path).to_dict("records"):
            audio_path = split_dir / item["audio_file"]
            image_path = split_dir / item["image_file"]
            if skip_missing_files and not (audio_path.exists() and image_path.exists()):
                continue
            rows.append({"split_name": split_name, "audio_path": str(audio_path), "image_path": str(image_path), "label": item["category"], "label_id": LABEL_TO_ID[item["category"]]})
    return pd.DataFrame(rows)

def read_wave(path):
    try:
        import torchaudio
        waveform, sample_rate = torchaudio.load(path)
        return waveform.mean(dim=0, keepdim=True), int(sample_rate)
    except Exception:
        with wave.open(str(path), "rb") as handle:
            sample_rate = handle.getframerate()
            channels = handle.getnchannels()
            width = handle.getsampwidth()
            frames = handle.readframes(handle.getnframes())
        dtype = np.int16 if width == 2 else np.uint8
        audio = np.frombuffer(frames, dtype=dtype).astype("float32")
        if channels > 1:
            audio = audio.reshape(-1, channels).mean(axis=1)
        audio = audio / 32768.0 if width == 2 else (audio - 128.0) / 128.0
        return torch.from_numpy(audio).unsqueeze(0), sample_rate

def resample_waveform(waveform, src_rate, dst_rate):
    if src_rate == dst_rate:
        return waveform.float()
    try:
        import torchaudio
        return torchaudio.transforms.Resample(orig_freq=src_rate, new_freq=dst_rate, lowpass_filter_width=64, rolloff=0.9475937167399596, resampling_method="sinc_interp_kaiser")(waveform.float())
    except Exception:
        from scipy.signal import resample_poly
        gcd = math.gcd(src_rate, dst_rate)
        y = resample_poly(waveform.numpy(), dst_rate // gcd, src_rate // gcd, axis=-1)
        return torch.from_numpy(y.copy()).float()

def apply_spectral_gate(waveform, cfg):
    if (not cfg.spectral_gate) or waveform.shape[-1] < cfg.n_fft:
        return waveform
    window = torch.hann_window(cfg.n_fft, device=waveform.device)
    spec = torch.stft(waveform, n_fft=cfg.n_fft, hop_length=cfg.hop_length, win_length=cfg.n_fft, window=window, return_complex=True)
    magnitude = spec.abs()
    noise = torch.quantile(magnitude, cfg.spectral_gate_noise_percentile / 100.0, dim=-1, keepdim=True)
    gated_mag = (magnitude - cfg.spectral_gate_strength * noise).clamp_min(0.0)
    phase = spec / magnitude.clamp_min(1e-8)
    return torch.istft(gated_mag * phase, n_fft=cfg.n_fft, hop_length=cfg.hop_length, win_length=cfg.n_fft, window=window, length=waveform.shape[-1])

def crop_waveform(waveform, window_samples, mode):
    total = waveform.shape[-1]
    if total < window_samples:
        return F.pad(waveform, (0, window_samples - total))
    if total == window_samples:
        return waveform
    max_start = total - window_samples
    if mode == "random":
        start = int(torch.randint(0, max_start + 1, (1,)).item())
    elif mode == "energy":
        energy = waveform.pow(2).mean(dim=0, keepdim=True).unsqueeze(0)
        kernel = torch.ones(1, 1, window_samples, device=waveform.device)
        start = int(F.conv1d(energy, kernel).argmax(dim=-1).item())
    else:
        start = max_start // 2
    return waveform[..., start:start + window_samples]

def waveform_to_mel(waveform, cfg):
    import torchaudio
    mel = torchaudio.transforms.MelSpectrogram(sample_rate=cfg.target_sample_rate, n_fft=cfg.n_fft, hop_length=cfg.hop_length, n_mels=cfg.n_mels, power=2.0)(waveform)
    mel = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=80)(mel)
    if cfg.normalize_audio_db:
        mel = (mel + 80.0) / 80.0
        mel = mel.clamp(0.0, 1.0)
    return mel

class AudioVisualDataset(Dataset):
    def __init__(self, frame, cfg, train=False):
        self.frame = frame.reset_index(drop=True)
        self.cfg = cfg
        self.crop_mode = cfg.train_crop if train else cfg.eval_crop
        self.image_transform = transforms.Compose([
            transforms.Resize((cfg.image_size, cfg.image_size)),
            transforms.RandomHorizontalFlip(p=0.5 if train else 0.0),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        waveform, sample_rate = read_wave(row.audio_path)
        waveform = resample_waveform(waveform, sample_rate, self.cfg.target_sample_rate)
        waveform = apply_spectral_gate(waveform, self.cfg)
        waveform = crop_waveform(waveform, int(round(self.cfg.target_sample_rate * self.cfg.audio_window_sec)), self.crop_mode)
        mel = waveform_to_mel(waveform, self.cfg)
        image = self.image_transform(Image.open(row.image_path).convert("RGB"))
        return {"waveform": waveform.squeeze(0), "audio": mel, "image": image, "label": torch.tensor(row.label_id, dtype=torch.long)}

def extract_model_embedding(output):
    if torch.is_tensor(output):
        return output
    pooler = getattr(output, "pooler_output", None)
    if torch.is_tensor(pooler):
        return pooler
    last_hidden = getattr(output, "last_hidden_state", None)
    if torch.is_tensor(last_hidden):
        return last_hidden[:, 0]
    if isinstance(output, (tuple, list)):
        for item in output:
            if torch.is_tensor(item):
                return item[:, 0] if item.ndim == 3 else item
            nested = extract_model_embedding(item)
            if torch.is_tensor(nested):
                return nested
    raise TypeError(f"Could not extract embedding from {type(output)!r}")

class PretrainedFusionBackbone(nn.Module):
    def __init__(self, data_cfg, model_cfg):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel, AutoProcessor, ClapModel
        self.sample_rate = data_cfg.target_sample_rate
        self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained(model_cfg.ast_model_name)
        self.ast_model = ASTModel.from_pretrained(model_cfg.ast_model_name)
        self.clap_processor = AutoProcessor.from_pretrained(model_cfg.clap_model_name)
        self.clap_model = ClapModel.from_pretrained(model_cfg.clap_model_name)
        self.ast_sample_rate = getattr(self.ast_feature_extractor, "sampling_rate", self.sample_rate)
        self.clap_sample_rate = getattr(getattr(self.clap_processor, "feature_extractor", None), "sampling_rate", self.sample_rate)
        self.image_model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        vit_dim = self.image_model.heads.head.in_features
        self.image_model.heads = nn.Identity()
        self.ast_dim = self.ast_model.config.hidden_size
        self.clap_dim = self.clap_model.config.projection_dim
        self.vit_dim = vit_dim
        if model_cfg.freeze_pretrained:
            for module in (self.ast_model, self.clap_model, self.image_model):
                for param in module.parameters():
                    param.requires_grad = False
    def processor_arrays(self, waveform, target_rate):
        arrays = [item.detach().float().cpu().numpy() for item in waveform]
        if target_rate == self.sample_rate:
            return arrays
        from scipy.signal import resample_poly
        gcd = math.gcd(self.sample_rate, target_rate)
        return [resample_poly(item, target_rate // gcd, self.sample_rate // gcd).astype("float32") for item in arrays]
    def encode_ast_mel(self, mel):
        if mel.ndim == 3:
            mel = mel.unsqueeze(1)
        mel_db = mel.squeeze(1)
        if mel_db.min() >= 0.0 and mel_db.max() <= 1.0:
            mel_db = mel_db * 80.0 - 80.0
        input_values = mel_db.transpose(1, 2)
        max_length = int(getattr(self.ast_feature_extractor, "max_length", input_values.shape[1]))
        if input_values.shape[1] > max_length:
            input_values = input_values[:, :max_length, :]
        elif input_values.shape[1] < max_length:
            pad = input_values.new_zeros(input_values.shape[0], max_length - input_values.shape[1], input_values.shape[2])
            input_values = torch.cat([input_values, pad], dim=1)
        mean = float(getattr(self.ast_feature_extractor, "mean", 0.0))
        std = float(getattr(self.ast_feature_extractor, "std", 1.0))
        input_values = (input_values - mean) / max(std, 1e-8)
        return extract_model_embedding(self.ast_model(input_values=input_values.to(mel.device)))
    def encode_clap(self, waveform):
        arrays = self.processor_arrays(waveform, self.clap_sample_rate)
        inputs = self.clap_processor(audio=arrays, sampling_rate=self.clap_sample_rate, return_tensors="pt", padding=True)
        inputs = {key: value.to(waveform.device) for key, value in inputs.items()}
        return extract_model_embedding(self.clap_model.get_audio_features(**inputs))
    def encode_all(self, waveform, audio, image):
        return self.encode_ast_mel(audio), self.encode_clap(waveform), self.image_model(image)

class EarlyFusionConcatNet(nn.Module):
    def __init__(self, data_cfg, model_cfg):
        super().__init__()
        self.backbone = PretrainedFusionBackbone(data_cfg, model_cfg)
        d = model_cfg.fusion_dim
        self.ast_proj = nn.Linear(self.backbone.ast_dim, d)
        self.clap_proj = nn.Linear(self.backbone.clap_dim, d)
        self.image_proj = nn.Linear(self.backbone.vit_dim, d)
        self.classifier = nn.Sequential(nn.LayerNorm(d * 3), nn.Linear(d * 3, d * 2), nn.GELU(), nn.Dropout(model_cfg.fusion_dropout), nn.Linear(d * 2, d), nn.GELU(), nn.Dropout(model_cfg.fusion_dropout), nn.Linear(d, len(LABELS)))
    def forward(self, waveform=None, image=None, audio=None):
        ast_emb, clap_emb, image_emb = self.backbone.encode_all(waveform, audio, image)
        return self.classifier(torch.cat([self.ast_proj(ast_emb), self.clap_proj(clap_emb), self.image_proj(image_emb)], dim=-1))

class MiddleFusionTransformerNet(nn.Module):
    def __init__(self, data_cfg, model_cfg):
        super().__init__()
        self.backbone = PretrainedFusionBackbone(data_cfg, model_cfg)
        d = model_cfg.fusion_dim
        self.ast_proj = nn.Linear(self.backbone.ast_dim, d)
        self.clap_proj = nn.Linear(self.backbone.clap_dim, d)
        self.image_proj = nn.Linear(self.backbone.vit_dim, d)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))
        layer = nn.TransformerEncoderLayer(d_model=d, nhead=model_cfg.fusion_heads, dim_feedforward=d * 4, dropout=model_cfg.fusion_dropout, activation="gelu", batch_first=True, norm_first=False)
        self.fusion = nn.TransformerEncoder(layer, num_layers=model_cfg.fusion_layers)
        self.classifier = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Dropout(model_cfg.fusion_dropout), nn.Linear(d, len(LABELS)))
    def forward(self, waveform=None, image=None, audio=None):
        ast_emb, clap_emb, image_emb = self.backbone.encode_all(waveform, audio, image)
        tokens = torch.stack([self.ast_proj(ast_emb), self.clap_proj(clap_emb), self.image_proj(image_emb)], dim=1)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        fused = self.fusion(torch.cat([cls, tokens], dim=1))
        return self.classifier(fused[:, 0])

class LateFusionLogitNet(nn.Module):
    def __init__(self, data_cfg, model_cfg):
        super().__init__()
        self.backbone = PretrainedFusionBackbone(data_cfg, model_cfg)
        d = model_cfg.fusion_dim
        self.ast_proj = nn.Linear(self.backbone.ast_dim, d)
        self.clap_proj = nn.Linear(self.backbone.clap_dim, d)
        self.audio_cls = nn.Parameter(torch.zeros(1, 1, d))
        audio_layer = nn.TransformerEncoderLayer(d_model=d, nhead=model_cfg.fusion_heads, dim_feedforward=d * 4, dropout=model_cfg.fusion_dropout, activation="gelu", batch_first=True, norm_first=False)
        self.audio_fusion = nn.TransformerEncoder(audio_layer, num_layers=model_cfg.fusion_layers)
        self.audio_classifier = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Dropout(model_cfg.fusion_dropout), nn.Linear(d, len(LABELS)))
        self.image_classifier = nn.Sequential(nn.LayerNorm(self.backbone.vit_dim), nn.Linear(self.backbone.vit_dim, d), nn.GELU(), nn.Dropout(model_cfg.fusion_dropout), nn.Linear(d, len(LABELS)))
        self.logit_weights = nn.Parameter(torch.zeros(2))
    def forward(self, waveform=None, image=None, audio=None):
        ast_emb, clap_emb, image_emb = self.backbone.encode_all(waveform, audio, image)
        audio_tokens = torch.stack([self.ast_proj(ast_emb), self.clap_proj(clap_emb)], dim=1)
        cls = self.audio_cls.expand(audio_tokens.size(0), -1, -1)
        audio_fused = self.audio_fusion(torch.cat([cls, audio_tokens], dim=1))[:, 0]
        audio_logits = self.audio_classifier(audio_fused)
        image_logits = self.image_classifier(image_emb)
        weights = torch.softmax(self.logit_weights, dim=0)
        return weights[0] * audio_logits + weights[1] * image_logits

def build_model(kind, data_cfg, model_cfg):
    if kind == "early":
        return EarlyFusionConcatNet(data_cfg, model_cfg)
    if kind == "middle":
        return MiddleFusionTransformerNet(data_cfg, model_cfg)
    if kind == "late":
        return LateFusionLogitNet(data_cfg, model_cfg)
    raise ValueError(kind)

def make_class_weight_tensor(frame, device):
    counts = frame["label_id"].value_counts().reindex(range(len(LABELS))).fillna(0).to_numpy(dtype=np.float32)
    weights = counts.sum() / np.maximum(counts, 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=device)

def compute_metrics(y_true, y_pred):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(LABELS))), average=PAPER_AVERAGE, zero_division=0)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(LABELS))), average="macro", zero_division=0)
    binary_true = (np.asarray(y_true) != LABEL_TO_ID["ambient"]).astype(int)
    binary_pred = (np.asarray(y_pred) != LABEL_TO_ID["ambient"]).astype(int)
    _, _, binary_f1, _ = precision_recall_fscore_support(binary_true, binary_pred, average="binary", zero_division=0)
    return {"paper_f1": float(f1), "paper_precision": float(precision), "paper_recall": float(recall), "macro_f1": float(macro_f1), "macro_precision": float(macro_precision), "macro_recall": float(macro_recall), "accuracy": float(accuracy_score(y_true, y_pred)), "binary_contact_f1": float(binary_f1)}

def run_epoch(model, loader, criterion, device, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")
    total_loss = 0.0
    total_items = 0
    logits_list = []
    labels_list = []
    for batch in tqdm(loader, leave=False):
        waveform = batch["waveform"].to(device)
        audio = batch["audio"].to(device)
        image = batch["image"].to(device)
        labels = batch["label"].to(device)
        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=device == "cuda"):
                logits = model(waveform=waveform, image=image, audio=audio)
                loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * labels.size(0)
        total_items += labels.size(0)
        logits_list.append(logits.detach().cpu())
        labels_list.append(labels.detach().cpu())
    logits = torch.cat(logits_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    metrics = compute_metrics(labels.numpy(), logits.argmax(dim=1).numpy())
    metrics["loss"] = total_loss / max(total_items, 1)
    return metrics

data_root = find_data_root(DATASET_ROOT)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
data_cfg = DataConfig(data_root=data_root)
model_cfg = ModelConfig()
device = "cuda" if torch.cuda.is_available() else "cpu"
index = build_index(data_cfg.data_root, data_cfg.skip_missing_files)
train_df = index[index["split_name"] == "audio_visual_dataset_default"].reset_index(drop=True)
val_df = index[index["split_name"] == "audio_visual_dataset_robo_default"].reset_index(drop=True)
train_loader = DataLoader(AudioVisualDataset(train_df, data_cfg, train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(AudioVisualDataset(val_df, data_cfg, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print("DATA_ROOT =", data_cfg.data_root)
print("OUTPUT_DIR =", output_dir)
print("device =", device)
print("train/val =", len(train_df), len(val_df))

model = build_model(FUSION_KIND, data_cfg, model_cfg).to(device)
criterion = nn.CrossEntropyLoss(weight=make_class_weight_tensor(train_df, device))
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
best_f1 = -1.0
best_metrics = None
best_epoch = 0
epochs_without_improvement = 0
history = []
mode = f"fusion_{FUSION_KIND}"

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, device, optimizer)
    val_metrics = run_epoch(model, val_loader, criterion, device)
    history.append({"epoch": epoch, "train": train_metrics, "val": val_metrics})
    print(f"{mode} epoch={epoch} train_f1={train_metrics['paper_f1']:.4f} val_f1={val_metrics['paper_f1']:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}")
    if val_metrics["paper_f1"] > best_f1 + MIN_DELTA:
        best_f1 = val_metrics["paper_f1"]
        best_metrics = val_metrics
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({"mode": mode, "model": model.state_dict(), "labels": LABELS, "paper_average": PAPER_AVERAGE, "best_val_metrics": val_metrics, "fusion_kind": FUSION_KIND}, output_dir / f"best_{mode}_model.pt")
    else:
        epochs_without_improvement += 1
        if EARLY_STOPPING_PATIENCE and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"{mode} early stopping at epoch={epoch}; best_epoch={best_epoch} best_val_f1={best_f1:.4f}")
            break

result = {"group": "pretrained", "mode": mode, "fusion": FUSION_KIND, "encoder": "AST+CLAP+ViT", "classifier": "mlp_head", "pretrained": True, "frozen": True, "paper_average": PAPER_AVERAGE, "best_paper_f1": best_f1, "best_epoch": best_epoch, "early_stopping_patience": EARLY_STOPPING_PATIENCE, "min_delta": MIN_DELTA, "best_val_metrics": best_metrics, "weight_file": str(output_dir / f"best_{mode}_model.pt"), "history": history}
(output_dir / f"{mode}_results.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
row = {"group": result["group"], "mode": result["mode"], "encoder": result["encoder"], "fusion": result["fusion"], "classifier": result["classifier"], "pretrained": result["pretrained"], "frozen": result["frozen"], "best_epoch": result["best_epoch"], "weight_file": result["weight_file"]}
row.update(best_metrics or {})
results = pd.DataFrame([row])
results.to_csv(output_dir / f"{mode}_single_result.csv", index=False)
results


DATA_ROOT = /kaggle/input/datasets/anhduy54/visual-audio/raw_dataset
OUTPUT_DIR = /kaggle/working/outputs/variants
device = cuda
train/val = 10676 2218


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

fusion_late epoch=19 train_f1=0.9934 val_f1=0.7515 val_macro_f1=0.6821


  0%|          | 0/2669 [00:00<?, ?it/s]

  0%|          | 0/555 [00:00<?, ?it/s]

fusion_late epoch=20 train_f1=0.9935 val_f1=0.7492 val_macro_f1=0.6729
fusion_late early stopping at epoch=20; best_epoch=10 best_val_f1=0.7975


,group,mode,encoder,fusion,classifier,pretrained,frozen,best_epoch,weight_file,paper_f1,paper_precision,paper_recall,macro_f1,macro_precision,macro_recall,accuracy,binary_contact_f1,loss
0,pretrained,fusion_late,AST+CLAP+ViT,late,mlp_head,True,True,10,/kaggle/working/outputs/variants/best_fusion_l...,0.797486,0.805027,0.809288,0.729262,0.745083,0.736534,0.809288,0.935002,1.35934
